# Validação do parser FTL

Este notebook valida materiais, seções, cotas, nós derivados, barras, apoios e cargas dos três arquivos de ponte. A implementação segue a topologia implícita do FTool: os nós são derivados dos endpoints das barras `2 1` e `4 1`. Todas as grandezas são normalizadas para SI (`N`, `m`, `Pa`).

In [ ]:
from math import isclose
from pathlib import Path

from src.parser import FtlParser
from src.visualizer import plot_ftool_model

INPUT_DIR = Path("inputs")
files = sorted(INPUT_DIR.glob("ponte_*.ftl"))
assert len(files) == 3, f"Esperava 3 arquivos, encontrei {len(files)}"

parsers = {path.name: FtlParser(path) for path in files}
models = {name: parser.parse() for name, parser in parsers.items()}
list(models)

## Testes de regressão

As contagens abaixo são snapshots dos arquivos atuais. As demais verificações testam invariantes estruturais e impedem a regressão para o comportamento anterior, que não criava barras nem associava cargas.

In [ ]:
expected_counts = {
    "ponte_1.ftl": {"nodes": 7, "members": 11},
    "ponte_2.ftl": {"nodes": 8, "members": 13},
    "ponte_3.ftl": {"nodes": 8, "members": 12},
}

for name, model in models.items():
    expected = expected_counts[name]

    assert len(model.materials) == 1
    assert len(model.sections) == 2
    assert len(model.point_loads) == 1
    assert len(model.dimensions) == 7
    assert len(model.nodes) == expected["nodes"]
    assert len(model.members) == expected["members"]

    material = model.materials[0]
    assert material.name == "palito"
    assert isclose(material.elasticity, 7.35e9)
    assert isclose(material.weight, 10_000.0)

    assert [section.name for section in model.sections] == ["PALITO", "palito"]
    assert isclose(model.sections[0].area, 0.00784)
    assert isclose(model.sections[0].inertia, 0.0)
    assert isclose(model.sections[1].inertia, 0.00185)

    node_ids = {node.id for node in model.nodes}
    assert all(member.length > 0 for member in model.members)
    assert all(member.start_node_id in node_ids for member in model.members)
    assert all(member.end_node_id in node_ids for member in model.members)
    assert all(member.material_name == "palito" for member in model.members)
    assert all(member.section_name in {"PALITO", "palito"} for member in model.members)
    assert all(state in (0, 1, 2) for node in model.nodes for state in (node.support.ux, node.support.uy, node.support.rz))

    pinned = [node for node in model.nodes if node.support.is_pinned]
    rollers = [node for node in model.nodes if node.support.is_roller_x]
    loaded = [node for node in model.nodes if node.load is not None]
    assert len(pinned) == len(rollers) == len(loaded) == 1
    assert isclose(pinned[0].x, 3.0) and isclose(pinned[0].y, 2.25)
    assert isclose(rollers[0].x, 4.0) and isclose(rollers[0].y, 2.25)
    assert isclose(loaded[0].x, 3.5) and isclose(loaded[0].y, 2.25)
    assert isclose(loaded[0].load.fx, 0.0)
    assert isclose(loaded[0].load.fy, -367.88)
    assert isclose(loaded[0].load.moment, 0.0)

# parse() precisa ser idempotente na mesma instância.
model_again = parsers["ponte_1.ftl"].parse()
assert len(model_again.nodes) == 7
assert len(model_again.members) == 11
assert len(model_again.dimensions) == 7

print("Todos os testes passaram.")

## Resumo dos modelos

In [ ]:
for name, model in models.items():
    print(
        f"{name}: materiais={len(model.materials)}, seções={len(model.sections)}, "
        f"cargas={len(model.point_loads)}, cotas={len(model.dimensions)}, "
        f"nós={len(model.nodes)}, barras={len(model.members)}"
    )

## Inspeção detalhada

O método `debug()` mostra conectividade, comprimentos, propriedades, apoios e a carga associada a cada nó.

In [ ]:
parsers["ponte_3.ftl"].debug()

## Validação visual

In [ ]:
for name, model in models.items():
    print(name)
    plot_ftool_model(
        model,
        show_node_ids=True,
        show_member_ids=True,
        show_dimensions=False,
    )